In [1]:
#Preprocess script for MESSAGEix-Canada Output before validation process

import pandas as pd
import numpy as np

file_path = "MESSAGEix-CA_baseline_2025-03-24--12-41.xlsx"  
df_correction = pd.read_excel(file_path)

# Replace empty values in the 'region' column with 'Unknown'
df_correction['Region'] = df_correction['Region'].replace(np.nan, 'Unkown')
duplicates = df_correction[df_correction.duplicated()]

print("Duplicated rows:")
print(duplicates)

num_duplicates = duplicates.shape[0]
print(f"Number of duplicated rows: {num_duplicates}")

df_correction = df_correction.drop_duplicates()


Duplicated rows:
              2025        2030        2035        2040        2045  \
2336  2.264993e+02  255.327047  261.365086  252.205858  237.894102   
2440  4.072005e+01   42.644683   34.867296   30.150096   26.369723   
2544  0.000000e+00    0.000000    0.000000    0.000000    0.000000   
2648  1.045242e+01    7.329652    7.066787    6.851688    6.596899   
2752  2.791386e+01   19.638455   13.263097   11.478990   10.000208   
2856  4.569387e+00    3.957415    4.116867    4.562182    4.249783   
2960  5.167273e-01    0.727076    1.069696    1.271176    1.137592   
3064  4.331329e+00    5.392537    6.964003    7.047483    6.790001   
3168 -5.204170e-18    0.082452    0.298438    0.327925    0.336158   
3272  1.553928e+02  156.674032  151.696021  140.837392  125.245383   
3376  2.255141e-17    0.302319    0.761771    0.841668    0.937628   
3480  6.213626e+01   59.775883   49.579951   39.744706   34.001909   
3584  3.120980e+01   33.021464   32.941331   28.352841   26.540930   
368

In [2]:
import pyam
import pandas as pd
from datetime import datetime

data = pyam.IamDataFrame(df_correction)
ranges_df = pd.read_csv('source_data.csv')

ranges_df = ranges_df.dropna(how='all')
# Define the acceptable lower limit for each variable, year, region, and scenario
acceptable_ranges = {}
for _, row in ranges_df.iterrows():
    region = row['Region']
    variable = row['Variable']
    year = int(row['Year'])
    scenario = row['Scenario']
    lower = float(row['Lower'])
    unit = row['Unit'] 

    if region not in acceptable_ranges:
        acceptable_ranges[region] = {}
    if variable not in acceptable_ranges[region]:
        acceptable_ranges[region][variable] = {}
    if year not in acceptable_ranges[region][variable]:
        acceptable_ranges[region][variable][year] = {}
    acceptable_ranges[region][variable][year][scenario] = {'lower': lower, 'unit': unit}

# Define acceptable deviation percentages
deviation_percentage = {
    2020: 0.10,
    2030: 0.20,
    2040: 0.20,
    2050: 0.20
}

# Filter for specific years of interest
years_of_interest = [2025, 2030, 2040, 2050]

results = []

within_deviation_count = 0
outside_deviation_count = 0

for year in years_of_interest:
    for region in data.region:
        for variable in data.variable:
            for scenario in data.scenario:
                if region in acceptable_ranges and variable in acceptable_ranges[region]:
                    if year in acceptable_ranges[region][variable]:
                        if scenario in acceptable_ranges[region][variable][year]:
                            acceptable_range = acceptable_ranges[region][variable][year][scenario]
                            lower_limit = acceptable_range['lower']
                            lower_unit = acceptable_range['unit'] 

                            df_filtered = data.filter(year=year, region=region, variable=variable, scenario=scenario)
                            
                            if not df_filtered.empty:
                                row = df_filtered.data.iloc[0]
                                value = row['value']
                                unit = row['unit']  
                                
                                # Unit check
                                if unit != lower_unit:
                                    print(f"Warning: Unit mismatch for {variable} in {region}, {year}, {scenario}. "
                                          f"Message output unit: {unit}, Expected source unit: {lower_unit}.")

                                deviation = 0
                                if lower_limit == 0:
                                    if value==0:
                                        deviation = 0
                                    else:
                                        deviation = 100
                                elif value < lower_limit:
                                    deviation = (lower_limit - value) / lower_limit * 100
                                elif value > lower_limit:
                                    deviation = (value - lower_limit) / lower_limit * 100
                                    
                                acceptable_dev = deviation_percentage.get(year, 0) * 100
                                
                                if deviation > acceptable_dev:
                                    results.append({
                                        'Year': year,
                                        'Region': region,
                                        'Variable': variable,
                                        'Scenario': scenario,  
                                        'Value': value,
                                        'Deviation %': deviation,
                                        'Unit': lower_unit,  
                                        'Source-value': lower_limit
                                    })
                                    outside_deviation_count += 1
                                else:
                                    within_deviation_count += 1



timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M')
filename = f'Validation_results_{scenario}_{timestamp}.csv'
results_df = pd.DataFrame(results)
results_df.to_csv(filename, index=False)
print(results_df)
print(f"Values within acceptable deviation: {within_deviation_count}")
print(f"Values outside acceptable deviation: {outside_deviation_count}")


c:\Users\brend\anaconda3\envs\messageix\Lib\site-packages\pyam\utils.py:318: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  .stack(dropna=True)


     Year   Region                                           Variable  \
0    2025  Alberta                           Carbon Sequestration|CCS   
1    2025  Alberta                                       Final Energy   
2    2025  Alberta                           Final Energy|Electricity   
3    2025  Alberta                                 Final Energy|Gases   
4    2025  Alberta                              Final Energy|Hydrogen   
..    ...      ...                                                ...   
643  2050    Yukon                     Final Energy|Industry|Hydrogen   
644  2050    Yukon  Final Energy|Residential and Commercial|Electr...   
645  2050    Yukon                        Final Energy|Transportation   
646  2050    Yukon                       Secondary Energy|Electricity   
647  2050    Yukon                          Secondary Energy|Hydrogen   

     Scenario        Value  Deviation %       Unit  Source-value  
0    baseline     0.989485    86.691600  Mt CO2/yr      